# Week 8-D: Generator Interaction & Marginal Information Gain
## Attribution Features ($A_1 \dots A_6$), Teacher-Forced Masking, and $MIG_i$

Features:
- $R_q(i)$: Query relevance
- $R_y(i)$: Draft relevance
- $Red(i)$: Redundancy against current FULL set
- $MIG_i = R_q(i) + \alpha R_y(i) - \beta Red(i)$
- Attention slot interaction: Attention weights assigned by SARA to latent slots during draft decoding.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.week8.semantic import SemanticScorer
from src.week8.data import load_config, load_week8_split
from src.week8.marginal_information import compute_candidate_marginal_information

cfg = load_config()
split = "internal_val"
records, _, _ = load_week8_split(split, cfg)
scorer = SemanticScorer()
print(f"W8-D ready with {len(records)} queries.")


In [ ]:
# Compute MIG for candidate ranks 5..10 across queries
mig_rows = []
for r in records[:50]:
    qid = str(r["question_id"])
    q_text = r["question"]
    passages = r.get("passages", [])
    if len(passages) < 10:
        continue
    base_full = [p["text"] for p in passages[:4]]
    draft_answer = r.get("gold_answers", [""])[0]  # Or base draft
    
    for rank in range(5, 11):
        cand_text = passages[rank - 1]["text"]
        mig_dict = compute_candidate_marginal_information(
            query=q_text,
            draft_answer=draft_answer,
            current_full_texts=base_full,
            candidate_text=cand_text,
            semantic_scorer=scorer,
            alpha=0.5,
            beta=0.5,
        )
        mig_dict["question_id"] = qid
        mig_dict["candidate_rank"] = rank
        mig_rows.append(mig_dict)

df_mig = pd.DataFrame(mig_rows)
print(f"Computed MIG features for {len(df_mig)} query-candidate pairs.")
print(df_mig[["query_relevance", "draft_relevance", "redundancy_max", "mig_score"]].describe().T[["mean", "std", "min", "max"]])
